In [ ]:
# Run this in the first cell
!pip install transformers datasets torch evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import files
import pandas as pd
import io

print("Click the button below to upload your CSV file:")

# 1. This creates a "Choose Files" button on your screen
uploaded = files.upload()

# 2. This automatically grabs the exact name of the file you uploaded
filename = list(uploaded.keys())[0]
print(f"\n✅ Successfully uploaded: '{filename}'")

# 3. This reads the file directly into your dataset (df)
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f"✅ Dataset loaded! Total rows: {len(df)}")

Click the button below to upload your CSV file:


Saving fake reviews dataset 50-50.csv to fake reviews dataset 50-50 (1).csv

✅ Successfully uploaded: 'fake reviews dataset 50-50 (1).csv'
✅ Dataset loaded! Total rows: 40432


In [ ]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

# NOTE: We assume 'df' is already loaded from your upload block!

# 1. Clean Data: Drop rows where text or label is missing
df = df.dropna(subset=['text_', 'label'])

# 2. Map Labels: 'CG' (Fake) -> 1, 'OR' (Real) -> 0
df['label'] = df['label'].map({'CG': 1, 'OR': 0})

# 3. Rename 'text_' to 'text'
# (Hugging Face Transformers require the column to be exactly named 'text')
df = df.rename(columns={'text_': 'text'})

# 4. Keep ONLY text and label columns to save memory on the GPU
# (We drop 'rating' and 'category' for this specific text model)
df = df[['text', 'label']]

# 5. Split Data (80% Train, 20% Test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# 6. Convert Pandas DataFrames into Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

print(f"✅ Training rows ready: {len(train_dataset)}")
print(f"✅ Testing rows ready: {len(test_dataset)}")

✅ Training rows ready: 32345
✅ Testing rows ready: 8087


In [ ]:
from transformers import AutoTokenizer

# Load the DistilBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Function to tokenize the text
def tokenize_function(examples):
    # Truncate cuts long reviews to 512 tokens (BERT's max limit)
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizing data... (This takes a few seconds)")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing data... (This takes a few seconds)


Map:   0%|          | 0/32345 [00:00<?, ? examples/s]

Map:   0%|          | 0/8087 [00:00<?, ? examples/s]

In [ ]:
import evaluate
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Load the pre-trained DistilBERT model
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# 2. Define how we want to measure success (Accuracy & F1)
metric_f1 = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"accuracy": acc, "f1": f1}

# 3. Set Training Parameters
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# 4. Create the Trainer (FIXED: Removed 'tokenizer=tokenizer')
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# 5. START TRAINING!
print("🚀 Starting Deep Learning Training...")
trainer.train()
print("✅ Training Complete!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Starting Deep Learning Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.090238,0.118818,0.968468,0.969199
2,0.037024,0.105764,0.975640,0.976060


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


✅ Training Complete!


In [ ]:
# Create a text classification pipeline with your newly trained model
from transformers import pipeline

classifier = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer, device=0)

my_reviews = [
    "This product is optimal and exceeds parameter expectations seamlessly.", # AI style
    "Honestly, it's okay but the battery died after 2 days. Kinda mad about it." # Human style
]

for review in my_reviews:
    result = classifier(review)[0]
    label_name = "FAKE (CG)" if result['label'] == 'LABEL_1' else "REAL (OR)"
    print(f"Text: {review}")
    print(f"Prediction: {label_name} with {result['score']*100:.2f}% confidence\n")

Text: This product is optimal and exceeds parameter expectations seamlessly.
Prediction: REAL (OR) with 99.98% confidence

Text: Honestly, it's okay but the battery died after 2 days. Kinda mad about it.
Prediction: REAL (OR) with 99.98% confidence



In [ ]:
from transformers import pipeline

# Create a prediction pipeline
classifier = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer, device=0)

# --- THE CORRECT TEST CASES FOR YOUR SPECIFIC DATASET ---
my_reviews = [
    # 1. The "Glitchy/Repetitive" Fake (Like the ones in your dataset)
    "I really like this product. This product is a good product because the product works well for the product.",

    # 2. The "Context Confusion" Fake (Older AI loses track of what it's talking about)
    "I bought these headphones to listen to music. The taste is very delicious and I will eat it again for dinner.",

    # 3. A normal Human Review (Should be Real)
    "Honestly, it's okay but the battery died after 2 days. Kinda mad about it.",

    # 4. Another normal Human Review (Should be Real)
    "Great item, shipping was fast. No complaints."
]

print("\n--- LIVE PREDICTIONS ---")
for review in my_reviews:
    result = classifier(review)[0]
    # LABEL_1 is CG (Fake), LABEL_0 is OR (Real)
    label_name = "🔴 FAKE (CG)" if result['label'] == 'LABEL_1' else "🟢 REAL (OR)"

    print(f"Text: '{review}'")
    print(f"Prediction: {label_name} with {result['score']*100:.2f}% confidence\n")


--- LIVE PREDICTIONS ---
Text: 'I really like this product. This product is a good product because the product works well for the product.'
Prediction: 🔴 FAKE (CG) with 99.77% confidence

Text: 'I bought these headphones to listen to music. The taste is very delicious and I will eat it again for dinner.'
Prediction: 🔴 FAKE (CG) with 99.85% confidence

Text: 'Honestly, it's okay but the battery died after 2 days. Kinda mad about it.'
Prediction: 🟢 REAL (OR) with 99.98% confidence

Text: 'Great item, shipping was fast. No complaints.'
Prediction: 🟢 REAL (OR) with 99.92% confidence



In [ ]:
from transformers import pipeline
# Create a prediction pipeline (Assuming your model is still loaded)
classifier = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer, device=0)

# --- 20 TEST EXAMPLES ---
my_reviews = [
    # ---- EXPECTED: FAKE (CG - Glitchy AI) ----
    "I bought this product and the product is a great product because I like the product.",
    "This is a movie that I watched and the movie was a film that I watched on the TV.",
    "The screen resolution on this laptop is amazing. It tastes so sweet and I added salt.", # Context loss
    "I am very happy with the purchase but I will never buy it again because it is the best.", # Contradiction
    "The book is a book that I read because reading is what you do with a book.",
    "My husband bought this dress for himself and she loves how the truck drives.", # Entity confusion
    "very Good. product. i am happy. with It! buy Again", # Punctuation glitch
    "It was a good day to be a good day and it was a good day for the item.",
    "The item has high quality. I am a person who likes this item very much. Five stars for the item.",
    "When I opened the box, the action, the action and the action was very good.",

    # ---- EXPECTED: REAL (OR - Human Writing) ----
    "Got this for my sons bday. He loves the blue color but the controler feels a bit cheap.", # Typos
    "Honestly, it's pretty mid. Not worth 50 bucks but it gets the job done.", # Slang
    "I've been using this shampoo for 3 years now. It really helps with my dry scalp during the winter.", # Anecdote
    "Shipping was fast (ordered Tuesday, got it Thursday). The assembly instructions were terrible though.",
    "Works as advertised. No issues so far.",
    "Arrived completely shattered! The box was crushed. I want a refund ASAP.", # Emotion
    "The noise cancellation on these Sony headphones is unreal. I can't even hear the airplane engine.", # Specifics
    "EDIT: After 2 weeks of use, the zipper broke. Dropping my rating to 2 stars.", # Human behavior
    "Put this in our guest bathroom. Matches the decor perfectly and visitors always ask where I got it.",
    "10/10 would buy again." # Internet shorthand
]

print("\n" + "="*50)
print(" 🚀 RUNNING 20 LIVE PREDICTIONS")
print("="*50 + "\n")

correct_fakes = 0
correct_reals = 0

for i, review in enumerate(my_reviews):
    # Determine the TRUE label based on how we organized the list (First 10 are Fake, Last 10 are Real)
    true_label = "FAKE (CG)" if i < 10 else "REAL (OR)"

    # Run Model Prediction
    result = classifier(review)[0]
    predicted_label = "FAKE (CG)" if result['label'] == 'LABEL_1' else "REAL (OR)"

    # Check if model got it right
    is_correct = (true_label == predicted_label)
    if is_correct and i < 10: correct_fakes += 1
    if is_correct and i >= 10: correct_reals += 1

    icon = "✅" if is_correct else "❌"

    print(f"Test {i+1}:")
    print(f"Text:       '{review}'")
    print(f"Model Says: {predicted_label} ({result['score']*100:.1f}% confidence)")
    print(f"Actual:     {true_label} {icon}\n")
    print("-" * 50)

# Final Scorecard
print("\n" + "="*50)
print(" 📊 FINAL SCORECARD")
print("="*50)
print(f"Caught Fake Reviews: {correct_fakes} / 10")
print(f"Verified Real Reviews: {correct_reals} / 10")
print(f"Overall Accuracy on Test: {((correct_fakes + correct_reals)/20)*100}%")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



 🚀 RUNNING 20 LIVE PREDICTIONS

Test 1:
Text:       'I bought this product and the product is a great product because I like the product.'
Model Says: FAKE (CG) (66.2% confidence)
Actual:     FAKE (CG) ✅

--------------------------------------------------
Test 2:
Text:       'This is a movie that I watched and the movie was a film that I watched on the TV.'
Model Says: FAKE (CG) (100.0% confidence)
Actual:     FAKE (CG) ✅

--------------------------------------------------
Test 3:
Text:       'The screen resolution on this laptop is amazing. It tastes so sweet and I added salt.'
Model Says: REAL (OR) (100.0% confidence)
Actual:     FAKE (CG) ❌

--------------------------------------------------
Test 4:
Text:       'I am very happy with the purchase but I will never buy it again because it is the best.'
Model Says: REAL (OR) (87.8% confidence)
Actual:     FAKE (CG) ❌

--------------------------------------------------
Test 5:
Text:       'The book is a book that I read because reading 

In [ ]:
from transformers import pipeline
import pandas as pd

# 1. Initialize the prediction pipeline
# 'device=0' tells it to use the GPU for fast prediction
classifier = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer, device=0)

# 2. Select 10 random samples from the TEST set (data the model hasn't seen during training)
# We use the 'test_df' created in Cell 2
random_samples = test_df.sample(n=10)

print("="*80)
print(f"{'REVIEW TEXT (First 100 chars)':<60} | {'ACTUAL':<10} | {'PREDICTED'}")
print("="*80)

correct_count = 0

for _, row in random_samples.iterrows():
    text = row['text']
    # Actual label from dataset (0=OR, 1=CG)
    actual_label = "FAKE (CG)" if row['label'] == 1 else "REAL (OR)"

    # Model prediction
    result = classifier(text[:512])[0] # BERT handles max 512 characters well
    predicted_label = "FAKE (CG)" if result['label'] == 'LABEL_1' else "REAL (OR)"

    # Check if correct
    status = "✅" if actual_label == predicted_label else "❌"
    if actual_label == predicted_label:
        correct_count += 1

    # Print formatted output
    short_text = (text[:57] + '..') if len(text) > 57 else text.ljust(59)
    print(f"{short_text:<60} | {actual_label:<10} | {predicted_label} {status}")

print("="*80)
print(f"TOTAL ACCURACY ON THIS RANDOM BATCH: {correct_count}/10 ({correct_count*10}%)")
print("="*80)

REVIEW TEXT (First 100 chars)                                | ACTUAL     | PREDICTED
I bought this with the intention of using it to make a sa..  | FAKE (CG)  | FAKE (CG) ✅
Nice quick read with interesting story and plot. Heroine ..  | REAL (OR)  | REAL (OR) ✅
I have several Leatherman multi-tool sets and these are t..  | FAKE (CG)  | FAKE (CG) ✅
I should have listened to the reviews and bought the size..  | FAKE (CG)  | FAKE (CG) ✅
Amazing for my pug with sensitive skin and sensitive eyes..  | FAKE (CG)  | FAKE (CG) ✅
Great product, made in the USA. Great Product!Very good q..  | FAKE (CG)  | FAKE (CG) ✅
?This was like seeing an old movie, with the characters b..  | FAKE (CG)  | FAKE (CG) ✅
smaller than they appear but it worked out for me because..  | REAL (OR)  | REAL (OR) ✅
An excellent novel, great companion book to The Lord of t..  | FAKE (CG)  | FAKE (CG) ✅
This movie is so bad that it's hard to watch it.  There i..  | FAKE (CG)  | FAKE (CG) ✅
TOTAL ACCURACY ON THIS RANDOM BATC